# A pedagogical implementation of SDDP

Stochastic Dual Dynamic Programming (Pereira and Pinto, 1991) is the method of choice for
multistage stochastic linear programs with many stages. Production codes such as
[SDDP.jl](https://odow.github.io/SDDP.jl/) hide the machinery behind a modelling API; the
purpose of this notebook is the opposite one: we write every ingredient of the algorithm
explicitly, in less than 150 lines of Julia, and we check each of them numerically.

The notebook follows the notation of the slides *07. SDDP*:

- $NLSD(t,k)$ is the nested L-shaped decomposition subproblem at stage $t$ for realization $k$;
- the forward pass **simulates** a scenario and produces trial states $\hat{x}_t$;
- the backward pass produces **optimality cuts** $E_{t-1} x_{t-1} + \theta_{t-1} \geq e_{t-1}$
  from the dual multipliers $\pi_{t,k}$;
- the objective of $NLSD(1,1)$ is a **deterministic lower bound**, the cost of the simulated
  scenarios is a **statistical upper bound**.

We use a hydro-thermal scheduling problem, the historical application of SDDP, small enough
that the extensive form can be solved and used as ground truth.

In [ ]:
using JuMP
using HiGHS
using Random
using Statistics
using Printf
using Plots

solver = HiGHS.Optimizer

## 1. The multistage problem

A single reservoir feeds a hydro plant; a thermal plant covers what the hydro plant cannot.
At every stage $t = 1,\ldots,H$ we observe the inflow $\xi_t$, then decide how much water to
turbine ($g^h_t$), how much fuel to burn ($g^e_t$) and how much water to spill ($s_t$). The
state variable is the level of the reservoir $x_t$ at the end of stage $t$.

The dynamic programming recursion reads
$$
  V_t(x_{t-1}, \xi_t) = \min_{x_t, g^h_t, g^e_t, s_t}\
      c_t g^e_t + \gamma s_t + \mathbb{E}_{\xi_{t+1}}\left[ V_{t+1}(x_t, \xi_{t+1}) \right]
$$
subject to
$$
\begin{align}
  x_t - x_{t-1} + g^h_t + s_t &= \xi_t   & (\text{water balance}) \\
  g^h_t + g^e_t &= d_t                   & (\text{demand}) \\
  0 \leq x_t \leq \overline{x},\quad 0 \leq g^e_t \leq \overline{g},\quad g^h_t, s_t &\geq 0
\end{align}
$$
with $V_{H+1} \equiv 0$. Writing $\mathcal{Q}_{t+1}(x_t) = \mathbb{E}_{\xi_{t+1}}[V_{t+1}(x_t, \xi_{t+1})]$,
the *expected cost-to-go*, the whole difficulty is that $\mathcal{Q}_{t+1}$ is unknown. SDDP builds
an outer approximation of it as a maximum of affine functions — the cuts.

The inflows $\xi_2, \ldots, \xi_H$ are discrete, independent from stage to stage
(**stagewise independence**), and $\xi_1$ is deterministic, so that $NLSD(1,1)$ is a single
deterministic problem, as in the slides.

In [ ]:
struct HydroInstance
    H::Int                            # number of stages
    capacity::Float64                 # reservoir capacity  x̄
    x0::Float64                       # initial reservoir level
    demand::Vector{Float64}           # demand d_t, per stage
    thermal_cost::Vector{Float64}     # unit cost c_t of thermal generation, per stage
    thermal_cap::Float64              # thermal capacity ḡ
    spill_cost::Float64               # unit penalty γ for spilling
    inflows::Vector{Vector{Float64}}  # support of ξ_t, per stage
    probs::Vector{Vector{Float64}}    # associated probabilities, per stage
end

nrealizations(inst, t) = length(inst.inflows[t])
nscenarios(inst) = prod(nrealizations(inst, t) for t in 1:inst.H)

In [ ]:
H = 6

inst = HydroInstance(
    H,
    250.0,                                        # reservoir capacity
    120.0,                                        # initial level
    [150.0, 180.0, 200.0, 170.0, 160.0, 150.0],   # demand
    [10.0, 12.0, 15.0, 18.0, 14.0, 11.0],         # thermal cost
    250.0,                                        # thermal capacity
    1.0,                                          # spill penalty
    [t == 1 ? [60.0] : [0.0, 40.0, 80.0, 120.0] for t in 1:H],
    [t == 1 ? [1.0]  : [0.2, 0.3, 0.3, 0.2]      for t in 1:H],
)

println("Scenarios in the tree: ", nscenarios(inst))

Two remarks on the instance.

- The thermal capacity exceeds the demand and spilling is unbounded, so every subproblem is
  feasible whatever the incoming state: the problem has **relatively complete recourse** and
  no feasibility cut is ever needed. This is why the code below only implements optimality
  cuts. (Feasibility cuts are described in the slides; they are generated from the dual ray
  of an infeasible subproblem exactly as in the L-shaped method.)
- The thermal cost varies from stage to stage, which is what gives water a nontrivial value:
  it is worth storing water before an expensive stage.

## 2. Reference solution: the extensive form

With $M = 4$ realizations per stage and $H = 6$ stages there are $4^5 = 1024$ scenarios, so we
can still write the deterministic equivalent and solve it directly. This gives us the exact
optimal value $z^\star$, against which the SDDP bounds will be compared — a luxury we lose as
soon as $H$ grows.

The function below also accepts a starting stage `t0` and an initial level `x_start`; we will
reuse it later to compute the **exact** expected cost-to-go $\mathcal{Q}_{t_0}(x)$.

In [ ]:
function extensive_form(inst::HydroInstance; t0::Int = 1, x_start::Float64 = inst.x0)
    m = Model(solver)
    set_silent(m)
    cost = AffExpr(0.0)
    function build_subtree(t, xin, prob)
        t > inst.H && return
        for (k, ξ) in enumerate(inst.inflows[t])
            p = prob * inst.probs[t][k]
            xout  = @variable(m, lower_bound = 0.0, upper_bound = inst.capacity)
            hydro = @variable(m, lower_bound = 0.0)
            therm = @variable(m, lower_bound = 0.0, upper_bound = inst.thermal_cap)
            spill = @variable(m, lower_bound = 0.0)
            @constraint(m, xout - xin + hydro + spill == ξ)     # water balance
            @constraint(m, hydro + therm == inst.demand[t])     # demand
            add_to_expression!(cost,
                p * (inst.thermal_cost[t] * therm + inst.spill_cost * spill))
            build_subtree(t + 1, xout, p)                       # non-anticipativity is
        end                                                     # implicit in the tree
    end
    build_subtree(t0, x_start, 1.0)
    @objective(m, Min, cost)
    return m
end

In [ ]:
det = extensive_form(inst)
optimize!(det)
z_star = objective_value(det)

@printf("Extensive form: %d variables, %d constraints, optimal value %.4f\n",
        num_variables(det),
        num_constraints(det; count_variable_in_set_constraints = false),
        z_star)

The extensive form is only tractable because the horizon is short. The number of scenarios
grows as $M^{H-1}$:

In [ ]:
for h in [6, 12, 24, 52]
    @printf("H = %2d, M = 4:  %.3e scenarios\n", h, 4.0^(h - 1))
end

## 3. The stage subproblem $NLSD(t)$

Thanks to stagewise independence, all the nodes of a given stage share the same expected
cost-to-go, hence the same cuts: we keep **one JuMP model per stage** instead of one per node
of the tree. This is the "cut sharing" of the slides, and it is what makes the backward pass
scalable.

Two ingredients make the model reusable:

- the incoming state is a variable `xin` tied to the trial value $\hat{x}_{t-1}$ by the
  constraint `incoming: xin == x̂`. Changing the trial state is then just a right-hand side
  update, and — crucially — the dual $\pi$ of that constraint is the sensitivity of the
  optimal value with respect to $\hat{x}_{t-1}$, i.e. a subgradient;
- the realization $\xi_t$ enters the right-hand side of the water balance, so sampling a new
  inflow is also a right-hand side update.

The variable $\theta$ is the epigraph variable approximating $\mathcal{Q}_{t+1}(x_t)$; it is bounded
below (here by $0$, a valid lower bound on any future cost) and constrained from below by the
cuts collected so far. At the last stage $V_{H+1} \equiv 0$, so we simply force $\theta_H = 0$.

In [ ]:
struct Cut
    intercept::Float64   # the cut is  θ ≥ intercept + slope * x
    slope::Float64
    x̂::Float64           # trial state where the cut was generated
end

mutable struct StageProblem
    t::Int
    model::Model
    xin::VariableRef
    xout::VariableRef
    θ::VariableRef
    hydro::VariableRef
    therm::VariableRef
    spill::VariableRef
    incoming::ConstraintRef    # xin == x̂          (dual = subgradient in x̂)
    balance::ConstraintRef     # xout - xin + hydro + spill == ξ
    cuts::Vector{Cut}
    nsolves::Int               # to count the linear programs we solve
end

function build_stage(inst::HydroInstance, t::Int; θmin::Float64 = 0.0)
    sp = Model(solver)
    set_silent(sp)
    @variable(sp, xin)
    @variable(sp, 0 <= xout <= inst.capacity)
    @variable(sp, hydro >= 0)
    @variable(sp, 0 <= therm <= inst.thermal_cap)
    @variable(sp, spill >= 0)
    @variable(sp, θ >= θmin)                 # θ ≈ Q_{t+1}(xout), bounded below
    if t == inst.H
        set_upper_bound(θ, 0.0)              # V_{H+1} ≡ 0
    end
    @constraint(sp, incoming, xin == 0.0)                       # rhs set to x̂ later
    @constraint(sp, balance, xout - xin + hydro + spill == 0.0) # rhs set to ξ later
    @constraint(sp, demand, hydro + therm == inst.demand[t])
    @objective(sp, Min, inst.thermal_cost[t] * therm + inst.spill_cost * spill + θ)
    return StageProblem(t, sp, xin, xout, θ, hydro, therm, spill,
                        incoming, balance, Cut[], 0)
end

In [ ]:
function solve_stage!(sp::StageProblem, x̂::Float64, ξ::Float64)
    set_normalized_rhs(sp.incoming, x̂)     # trial state coming from stage t-1
    set_normalized_rhs(sp.balance, ξ)      # sampled inflow
    optimize!(sp.model)
    sp.nsolves += 1
    status = termination_status(sp.model)
    status == OPTIMAL ||
        error("stage $(sp.t): subproblem is not optimal (status: $status)")
    return (
        Q          = objective_value(sp.model),                    # stage cost + θ
        π          = dual(sp.incoming),                            # ∂Q / ∂x̂
        xout       = value(sp.xout),                               # new state
        stage_cost = objective_value(sp.model) - value(sp.θ),      # cost of stage t only
        hydro      = value(sp.hydro),
        therm      = value(sp.therm),
        spill      = value(sp.spill),
    )
end

Let us look at a subproblem before any cut has been added. Note how small it is compared to
the extensive form: SDDP replaces one huge linear program by many tiny ones.

In [ ]:
demo = build_stage(inst, 3)
println(demo.model)

## 4. From dual multipliers to cuts

Let
$$
  Q_t(\hat{x}, \xi) = \min \left\{ c_t g^e_t + \gamma s_t + \theta_t \ :\
      x_{t-1} = \hat{x},\ \text{(balance, demand, cuts)} \right\}
$$
be the optimal value of $NLSD(t)$ for the trial state $\hat{x}$ and the realization $\xi$, and
$\mathcal{Q}_t(\hat{x}) = \sum_k p_{t,k} Q_t(\hat{x}, \xi_{t,k})$ its expectation. Because the
subproblem is a linear program whose right-hand side depends linearly on $\hat{x}$, $Q_t(\cdot, \xi)$
is convex and piecewise linear, and the dual $\pi$ of the constraint `xin == x̂` satisfies
$\pi \in \partial Q_t(\hat{x}, \xi)$. Averaging,
$$
  \bar{\pi} = \sum_{k=1}^{M_t} p_{t,k}\, \pi_{t,k} \in \partial \mathcal{Q}_t(\hat{x}),
  \qquad
  \bar{Q} = \sum_{k=1}^{M_t} p_{t,k}\, Q_t(\hat{x}, \xi_{t,k}) = \mathcal{Q}_t(\hat{x}),
$$
and convexity gives the valid inequality, for every $x$,
$$
  \mathcal{Q}_t(x) \;\geq\; \bar{Q} + \bar{\pi} (x - \hat{x}).
$$
This is the cut we add to the subproblem of stage $t-1$, as a constraint on its epigraph
variable:
$$
  \theta_{t-1} \;\geq\; \bar{Q} + \bar{\pi}\,(x_{t-1} - \hat{x}).
$$
In the notation $E_{t-1} x_{t-1} + \theta_{t-1} \geq e_{t-1}$ of the slides,
$E_{t-1} = -\bar{\pi}$ and $e_{t-1} = \bar{Q} - \bar{\pi}\hat{x}$. The cut is **valid**
everywhere (it never cuts off the true value function) and **tight** at $\hat{x}$, which is why
the lower bound improves monotonically at the states that the forward pass visits.

In [ ]:
function add_cut!(sp::StageProblem, Q̄::Float64, π̄::Float64, x̂::Float64)
    push!(sp.cuts, Cut(Q̄ - π̄ * x̂, π̄, x̂))
    @constraint(sp.model, sp.θ >= Q̄ + π̄ * (sp.xout - x̂))
    return sp.cuts[end]
end

# the outer approximation carried by a stage problem, for plotting and checking
Vhat(sp::StageProblem, x::Real) =
    isempty(sp.cuts) ? lower_bound(sp.θ) :
    max(lower_bound(sp.θ), maximum(c.intercept + c.slope * x for c in sp.cuts))

Before going further, let us verify numerically that the dual really is the derivative we
claim. At the last stage there is no future cost, and with $\hat{x} = 50$, $\xi = 40$ the plant
turbines all the available water ($90$) and burns fuel for the remaining $60$ units of demand
at a cost of $11$ per unit; one extra unit of water in the reservoir saves exactly $11$.

In [ ]:
spH = build_stage(inst, inst.H)
x̂, ξ, ε = 50.0, 40.0, 1e-4
r0 = solve_stage!(spH, x̂, ξ)
r1 = solve_stage!(spH, x̂ + ε, ξ)

@printf("Q_H(%.1f, %.1f)    = %.4f\n", x̂, ξ, r0.Q)
@printf("dual π            = %.6f\n", r0.π)
@printf("finite difference = %.6f\n", (r1.Q - r0.Q) / ε)

The expected cut at a trial state requires one linear program per realization of $\xi_t$ —
this is the inner loop of the backward pass.

In [ ]:
function expected_cut(sp::StageProblem, inst::HydroInstance, x̂::Float64)
    Q̄, π̄ = 0.0, 0.0
    for (k, ξ) in enumerate(inst.inflows[sp.t])
        r = solve_stage!(sp, x̂, ξ)
        p = inst.probs[sp.t][k]
        Q̄ += p * r.Q          # Σ p_k Q_t(x̂, ξ_k)
        π̄ += p * r.π          # Σ p_k π_{t,k}
    end
    return Q̄, π̄
end

## 5. Forward pass

Nested decomposition would enumerate the whole tree; SDDP **samples** one scenario instead.
Starting from $x_0$ we solve $NLSD(1,1)$, then, for $t = 2,\ldots,H$, we draw an inflow and
solve $NLSD(t)$ with the state produced by the previous stage. The pass returns

- the objective of $NLSD(1,1)$, a valid **lower bound** on the optimal value (the cuts are an
  outer approximation of the true cost-to-go, so the first-stage problem is a relaxation);
- the cost of the sampled scenario, obtained by summing the *stage costs only* — $\theta$ must
  be excluded, otherwise we would count an approximation of the future twice;
- the trial states $\hat{x}_1, \ldots, \hat{x}_H$, where the backward pass will build cuts.

Each forward pass costs $H$ linear programs, whatever the size of the scenario tree.

In [ ]:
function sample_index(rng, p::Vector{Float64})
    u, c = rand(rng), 0.0
    for (k, pk) in enumerate(p)
        c += pk
        u <= c && return k
    end
    return length(p)
end

function forward_pass!(stages, inst::HydroInstance, rng)
    trial = zeros(inst.H)
    r = solve_stage!(stages[1], inst.x0, inst.inflows[1][1])   # NLSD(1,1), deterministic
    lb, cost, trial[1] = r.Q, r.stage_cost, r.xout
    for t in 2:inst.H
        k = sample_index(rng, inst.probs[t])                   # sample ξ_t
        r = solve_stage!(stages[t], trial[t-1], inst.inflows[t][k])
        cost += r.stage_cost
        trial[t] = r.xout
    end
    return (lb = lb, cost = cost, trial = trial)
end

## 6. Backward pass

We now walk back from $t = H$ to $t = 2$. At each stage, and for each trial state produced by
the forward passes, we solve $NLSD(t)$ for **all** the realizations of $\xi_t$, average the
optimal values and the duals, and add the resulting cut to stage $t-1$. Because of stagewise
independence the cut is added once and is valid for every node of stage $t-1$.

For $K$ trial sequences the backward pass costs $K \sum_{t=2}^{H} M_t$ linear programs — linear
in the horizon, instead of exponential.

In [ ]:
function backward_pass!(stages, inst::HydroInstance, trials)
    for t in inst.H:-1:2
        for trial in trials
            x̂ = trial[t-1]                              # state visited by the forward pass
            Q̄, π̄ = expected_cut(stages[t], inst, x̂)     # M_t linear programs
            add_cut!(stages[t-1], Q̄, π̄, x̂)             # cut shared by all nodes of stage t-1
        end
    end
    return
end

## 7. Bounds, and the stopping rule

The objective of $NLSD(1,1)$ is a deterministic lower bound $z_{LB}$; it increases as cuts
accumulate. The mean cost $\bar{z}_K$ of $K$ simulated scenarios estimates the expected cost of
the current policy, which is an upper bound on the optimal value — but only *in expectation*,
and it is noisy. The slides' criterion stops as soon as
$$
  z_{LB} \;\in\; \left( \bar{z}_K - z_{1-\alpha/2}\,\hat{s}/\sqrt{K},\;
                        \bar{z}_K + z_{1-\alpha/2}\,\hat{s}/\sqrt{K} \right),
$$
i.e. when the lower bound is no longer distinguishable from the simulated cost. We keep it,
but make it optional, so that we can also run a fixed number of iterations and *see* what the
criterion misses (section 11).

In [ ]:
const Φinv = Dict(0.90 => 1.6448536269514722,     # normal quantiles z_{1-α/2}
                  0.95 => 1.9599639845400540,
                  0.99 => 2.5758293035489004)

function sddp(inst::HydroInstance; iterations = 25, K = 5, α = 0.05,
              seed = 1234, verbose = true, stop_on_criterion = false)
    rng = MersenneTwister(seed)
    stages = [build_stage(inst, t) for t in 1:inst.H]
    z = Φinv[1 - α]
    history = NamedTuple[]
    verbose && @printf("%5s %11s %11s %11s %11s %7s\n",
                       "iter", "lower", "sim. mean", "ci low", "ci high", "cuts")
    for it in 1:iterations
        lb = 0.0
        costs = zeros(K)
        trials = Vector{Vector{Float64}}(undef, K)
        for i in 1:K                                    # K forward passes
            fp = forward_pass!(stages, inst, rng)
            lb, costs[i], trials[i] = fp.lb, fp.cost, fp.trial
        end
        z̄ = mean(costs)
        half = K > 1 ? z * std(costs) / sqrt(K) : 0.0
        ncuts = sum(length(sp.cuts) for sp in stages)
        push!(history, (iter = it, lb = lb, z̄ = z̄, half = half, ncuts = ncuts))
        verbose && @printf("%5d %11.3f %11.3f %11.3f %11.3f %7d\n",
                           it, lb, z̄, z̄ - half, z̄ + half, ncuts)
        if stop_on_criterion && lb >= z̄ - half
            verbose && println("Statistical stopping criterion met at iteration $it.")
            return stages, history
        end
        backward_pass!(stages, inst, trials)             # cuts for the next iteration
    end
    verbose && println("Iteration limit reached.")
    return stages, history
end

## 8. Running the algorithm

We run 25 iterations with $K = 5$ forward passes per iteration, without early stopping.

In [ ]:
stages, history = sddp(inst; iterations = 25, K = 5, stop_on_criterion = false);

In [ ]:
@printf("SDDP lower bound  : %.4f\n", history[end].lb)
@printf("Extensive form    : %.4f\n", z_star)
@printf("Relative gap      : %.2e\n", (z_star - history[end].lb) / z_star)

nlp = sum(sp.nsolves for sp in stages)
@printf("\nLinear programs solved: %d, each with %d variables\n",
        nlp, num_variables(stages[1].model))
@printf("Extensive form        : 1 linear program with %d variables\n", num_variables(det))

The lower bound reaches the true optimum to machine precision. The bookkeeping is the point of
the exercise: we solved a few thousand tiny linear programs instead of one large one — a trade
that pays off as soon as the tree becomes too large to be written down, which happens around
$H = 15$ here.

In [ ]:
its  = getfield.(history, :iter)
lbs  = getfield.(history, :lb)
ubs  = getfield.(history, :z̄)
half = getfield.(history, :half)

p1 = plot(its, lbs, lw = 2, label = "lower bound (NLSD(1,1))",
          xlabel = "iteration", ylabel = "cost", legend = :bottomright,
          title = "SDDP convergence")
plot!(p1, its, ubs, ribbon = half, lw = 1, color = :orange, fillalpha = 0.2,
      label = "simulated cost (95% CI)")
hline!(p1, [z_star], ls = :dash, color = :black, label = "true optimum")

The lower bound climbs quickly and then flattens; the simulated cost oscillates wildly around
it. That asymmetry is intrinsic: the lower bound is deterministic and monotone, the upper
bound is a Monte Carlo estimate whose standard error decreases only as $1/\sqrt{K}$.

## 9. Simulating the policy

Training produces cuts, i.e. a *policy*: at every stage, solve the subproblem with the current
cuts and apply the decision. Simulating it on fresh scenarios is the honest way to evaluate
it, and it is also how the statistical upper bound is obtained.

In [ ]:
function simulate_policy(stages, inst::HydroInstance, n; seed = 2024)
    rng = MersenneTwister(seed)
    costs  = zeros(n)
    levels = zeros(n, inst.H)
    therm  = zeros(n, inst.H)
    for i in 1:n
        r = solve_stage!(stages[1], inst.x0, inst.inflows[1][1])
        costs[i], levels[i, 1], therm[i, 1] = r.stage_cost, r.xout, r.therm
        for t in 2:inst.H
            k = sample_index(rng, inst.probs[t])
            r = solve_stage!(stages[t], levels[i, t-1], inst.inflows[t][k])
            costs[i]  += r.stage_cost
            levels[i, t], therm[i, t] = r.xout, r.therm
        end
    end
    return (costs = costs, levels = levels, therm = therm)
end

In [ ]:
n = 1000
sim = simulate_policy(stages, inst, n)

@printf("simulated policy cost : %.2f ± %.2f (95%% CI)\n",
        mean(sim.costs), 1.96 * std(sim.costs) / sqrt(n))
@printf("lower bound           : %.2f\n", history[end].lb)
@printf("true optimum          : %.2f\n", z_star)
println("\nmean reservoir level per stage : ", round.(vec(mean(sim.levels, dims = 1)), digits = 1))
println("mean thermal generation        : ", round.(vec(mean(sim.therm,  dims = 1)), digits = 1))

Note that the confidence interval on the simulated cost easily covers values *below* the
optimum: a Monte Carlo estimate of an upper bound is not an upper bound. Only the
deterministic lower bound is guaranteed.

The policy is readable in the averages: water is stored during the cheap stages 1–2 and
released during the expensive stages 3–4, and the reservoir is emptied at the horizon — the
usual artefact of a finite horizon without a terminal value function.

In [ ]:
p2 = plot(title = "Reservoir trajectories under the SDDP policy",
          xlabel = "stage", ylabel = "level at the end of the stage", legend = false)
for i in 1:40
    plot!(p2, 1:inst.H, sim.levels[i, :], alpha = 0.3, color = :steelblue)
end
plot!(p2, 1:inst.H, vec(mean(sim.levels, dims = 1)), lw = 3, color = :red)
p2

## 10. How good are the value functions?

SDDP does *not* approximate the cost-to-go everywhere: it refines it where the policy goes.
Here we can check that claim exactly, since $\mathcal{Q}_t(x)$ can be computed by solving the
extensive form of stages $t, \ldots, H$ starting from the level $x$.

In [ ]:
exact_Q(inst, t, x) = (m = extensive_form(inst; t0 = t, x_start = Float64(x));
                       optimize!(m); objective_value(m))

grid = 0.0:5.0:inst.capacity
gaps = [exact_Q(inst, 5, x) - Vhat(stages[4], x) for x in grid]

@printf("min of Q₅ - V̂₅ over the grid : %.2e   (≥ 0 up to tolerance: the cuts are valid)\n",
        minimum(gaps))
@printf("max of Q₅ - V̂₅ over the grid : %.3f\n", maximum(gaps))
println("states visited at stage 4    : ",
        sort(unique(round.([c.x̂ for c in stages[4].cuts], digits = 1) .+ 0.0)))

In [ ]:
plots = []
for t in 2:5
    Qex = [exact_Q(inst, t, x) for x in grid]
    Qap = [Vhat(stages[t-1], x) for x in grid]
    p = plot(grid, Qex, lw = 3, color = :black, label = "exact",
             title = "Q_$t", xlabel = "x", ylabel = "expected cost-to-go",
             legend = (t == 2 ? :topright : false))
    for c in stages[t-1].cuts                       # the individual cuts
        plot!(p, grid, c.intercept .+ c.slope .* grid,
              color = :gray, alpha = 0.25, lw = 1, label = "")
    end
    plot!(p, grid, Qap, lw = 2, color = :crimson, label = "max of cuts")
    ylims!(p, minimum(Qex) - 300, maximum(Qex) + 300)
    push!(plots, p)
end
plot(plots..., layout = (2, 2), size = (900, 650))

Every cut lies below the true function, and the approximation touches it only in the region
the forward passes actually visited — elsewhere it can be far off, and that is not a defect:
those states are irrelevant for the optimal policy. It is also why the value functions of an
SDDP model should never be used far away from the simulated trajectories.

## 11. The statistical stopping rule in action

Let us now enable the criterion of the slides.

In [ ]:
stages_b, history_b = sddp(inst; iterations = 25, K = 5, stop_on_criterion = true)

@printf("\nstopped with lb = %.3f, true optimum = %.3f, actual gap = %.2f%%\n",
        history_b[end].lb, z_star, 100 * (z_star - history_b[end].lb) / z_star)

The criterion fires after a handful of iterations, with a gap of the order of a percent still
open. The reason is visible in the log: with $K = 5$ the confidence interval is very wide, so
the lower bound enters it long before the policy is optimal. The rule is *optimistic*, and
increasing $K$ makes it more conservative but more expensive — the number of forward passes
needed for a $1\%$ gap at $95\%$ confidence is
$$
  K = \left( \frac{z_{1-\alpha/2}\, \hat{s}}{0.01\, \bar{z}_K} \right)^2 .
$$

In [ ]:
ŝ = std(sim.costs)
z̄ = mean(sim.costs)
@printf("ŝ = %.2f, z̄ = %.2f  ⟹  K ≈ %d forward passes for a 1%% gap at 95%%\n",
        ŝ, z̄, ceil(Int, (1.96 * ŝ / (0.01 * z̄))^2))

This is why modern implementations rely on the deterministic lower bound and on a simulation
performed *after* training, rather than on this criterion, and typically stop on an iteration
limit, a time limit, or a stalling bound.

## 12. Two stages: SDDP is the L-shaped method

With $H = 2$ the backward pass builds cuts for the first stage only, from the duals of all the
second-stage subproblems: this is exactly the single-cut L-shaped method of lecture 04, the
only difference being that the trial point comes from a master problem that is itself the
first stage.

In [ ]:
inst2 = HydroInstance(2, 250.0, 120.0, [150.0, 180.0], [10.0, 12.0], 250.0, 1.0,
                      [[60.0], [0.0, 40.0, 80.0, 120.0]],
                      [[1.0],  [0.2, 0.3, 0.3, 0.2]])

det2 = extensive_form(inst2)
optimize!(det2)

stages2, history2 = sddp(inst2; iterations = 6, K = 1)

@printf("\nSDDP lower bound = %.4f, extensive form = %.4f\n",
        history2[end].lb, objective_value(det2))
println("\noptimality cuts generated at the first stage:")
for c in stages2[1].cuts
    @printf("  θ ≥ %9.3f %+8.4f x   (generated at x̂ = %7.3f)\n",
            c.intercept, c.slope, c.x̂)
end

The last cuts are duplicates: once the first-stage solution stops moving, the forward pass
keeps returning the same trial state and the backward pass keeps rebuilding the same
inequality. Detecting and discarding such redundant cuts (*cut selection*) is one of the
practical ingredients of an efficient implementation.

## 13. What a production implementation adds

The 150 lines above contain the algorithm; what separates them from SDDP.jl is engineering
and generality:

- **feasibility cuts**, for problems without relatively complete recourse (our instance is
  always feasible by construction);
- **multi-cut variants**, keeping one $\theta$ per realization instead of averaging, and
  **cut selection** to keep the subproblems small;
- **risk measures** (AV@R and friends) in place of the expectation in the backward pass;
- **general policy graphs**: cyclic graphs for infinite horizon problems, Markovian noise when
  stagewise independence fails;
- **multidimensional states**, several reservoirs, integer variables (SDDiP), parallelism.

### Exercises

1. Replace the horizon by $H = 12$ and check that the extensive form becomes impractical while
   SDDP is barely affected. How does the cost per iteration grow?
2. Implement the multi-cut version: store $\theta_k$ per realization in stage $t-1$ and add
   $M_t$ cuts per trial state. Compare the number of iterations and the time per iteration.
3. Add a terminal value function, for instance $\theta_H \geq \lambda(\overline{x} - x_H)$ with
   $\lambda$ the price of water, and observe how the end-of-horizon emptying disappears.
4. Break stagewise independence by making the inflow of stage $t$ depend on that of stage
   $t-1$ (an autoregressive model). Show that the cuts can no longer be shared, and that
   adding the previous inflow to the state vector restores the algorithm.
5. Implement the feasibility cuts of the slides: remove the spill variable and reduce the
   thermal capacity below the demand, so that some incoming states become infeasible.